In [1]:
import os
import scvelo as scv
import scanpy as sc
import numpy as np
import anndata
import loompy
import pandas as pd
import argparse
import gc
import scipy.sparse as sp
import matplotlib.pyplot as plt

In [2]:
scv.settings.verbosity = 3
scv.settings.presenter_view = True
scv.set_figure_params('scvelo')

In [5]:
mainDir = "/data/ebaird/scRNAseq/bairdetal/"
repDir = mainDir + "scvelo/"
figDir = repDir + "figs/"
tabDir = repDir + "tables/"

os.makedirs(mainDir, exist_ok=True)
os.makedirs(repDir, exist_ok=True)
os.makedirs(figDir, exist_ok=True)
os.makedirs(tabDir, exist_ok=True)

In [ ]:
# merged_adata_file = "/data/ebaird/scRNAseqreports/res/Gal10d_Gal12d_Flp10d_Flp12d_070525/anndata_Gal10d_Gal12d_Flp10d_Flp12d.h5ad"
# merged_adata = anndata.read(merged_adata_file)

In [6]:
# Build raw counts adata from scratch
adata = sc.read_mtx(mainDir + "composition_DEG_signatures/counts_matrix.mtx")

print(f"Scanpy MTX shape: {adata.shape}")

# Load genes and barcodes
genes = pd.read_csv(mainDir + "composition_DEG_signatures/genes.csv", header=0)
barcodes = pd.read_csv(mainDir + "composition_DEG_signatures/barcodes.csv", header=0)
metadata = pd.read_csv(mainDir + "composition_DEG_signatures/metadata.csv", index_col=0)

# Set the dimensions based on the MTX shape
if adata.shape[0] == len(barcodes):  # If rows = barcodes
    print("MTX is already cells × genes")
    adata.obs_names = barcodes.iloc[:, 1].values
    adata.var_names = genes.iloc[:, 1].values
else:  # If rows = genes, need to transpose
    print("MTX is genes × cells, transposing...")
    adata = adata.T
    adata.obs_names = barcodes.iloc[:, 1].values
    adata.var_names = genes.iloc[:, 1].values

print(f"Final shape: {adata.shape}")

# Add metadata
adata.obs = adata.obs.join(metadata, how='left')

merged_adata = adata

# NEW: Load and integrate Seurat PCA
print("Loading Seurat PCA...")
seurat_pca = pd.read_csv(mainDir + "composition_DEG_signatures/seurat_pca.csv", index_col=0)

# NEW: Load and integrate Seurat UMAP
print("Loading Seurat UMAP...")
seurat_umap = pd.read_csv(mainDir + "composition_DEG_signatures/seurat_umap.csv", index_col=0)

# Ensure the cell barcodes match between datasets
print("Matching cell barcodes...")
common_cells = adata.obs_names.intersection(seurat_pca.index)
common_cells = common_cells.intersection(seurat_umap.index)  # Also check UMAP
print(f"Common cells: {len(common_cells)}")

if len(common_cells) < len(adata.obs_names):
    print(f"Warning: {len(adata.obs_names) - len(common_cells)} cells not found in Seurat embeddings")
    # Subset to common cells
    adata = adata[common_cells].copy()
    # Also subset metadata if needed
    if len(metadata) > len(common_cells):
        metadata = metadata.loc[common_cells]

# Reorder Seurat embeddings to match scVelo AnnData order
seurat_pca = seurat_pca.loc[adata.obs_names]
seurat_umap = seurat_umap.loc[adata.obs_names]

# Add Seurat PCA to AnnData object
adata.obsm['X_pca'] = seurat_pca.values

# NEW: Add Seurat UMAP to AnnData object
adata.obsm['X_umap'] = seurat_umap.values

print(f"Seurat PCA shape: {seurat_pca.shape}")
print(f"Seurat UMAP shape: {seurat_umap.shape}")
print(f"Added to adata.obsm['X_pca']: {adata.obsm['X_pca'].shape}")
print(f"Added to adata.obsm['X_umap']: {adata.obsm['X_umap'].shape}")

In [7]:
print("Data matrix shape:", merged_adata.X.shape)
print("Data type:", type(merged_adata.X))

# Check if data contains integers (typical for raw counts)
if hasattr(merged_adata.X, 'data'):
    sample_values = merged_adata.X.data[:100]  # Check first 100 non-zero values
else:
    sample_values = merged_adata.X.flatten()[:100]  # For dense matrices

print("Sample values:", sample_values)
print("Are values integers?", np.all(np.mod(sample_values, 1) == 0))

In [8]:
velocyto_loom_dir = "/data/ebaird/scRNAseq/SCENTINELsep24/cellranger/"
samples = ["flp_10d", "flp_12d", "gal_10d", "gal_12d"]
velocyto_loom_files = []

for sample in samples:
    loom_file_path = os.path.join(velocyto_loom_dir, sample, "velocyto", f"{sample}.loom")
    velocyto_loom_files.append(loom_file_path)

print("Constructed loom file paths:")
for file_path in velocyto_loom_files:
    print(file_path)

In [9]:
def close_loom_connection(loom_file_path):
    """Force close any existing connection to a loom file"""
    try:
        gc.collect()
        
        with loompy.connect(loom_file_path, mode='r') as temp:
            pass 
        print("✓ Loom connection closed")
    except Exception as e:
        print(f"Note: {e}")

In [10]:
for path in velocyto_loom_files:
    close_loom_connection(path)

In [11]:
def add_velocity_to_anndata(merged_anndata, loom_file, sample_name):    
    loom = loompy.connect(loom_file)
    
    # Get cell IDs from Loom and clean them
    loom_cells = loom.ca['CellID']
    print(f"Loom cells (raw) for sample {sample_name}: {loom_cells[:5]}... ({len(loom_cells)} total)")
    loom_cells_cleaned = [cell.split(':')[1].rstrip('x') for cell in loom_cells]
    print(f"Loom cells (cleaned) for sample {sample_name}: {loom_cells_cleaned[:5]}... ({len(loom_cells_cleaned)} total)")
    
    # Get full cell names from AnnData for the given sample
    merged_cells_full = merged_anndata.obs_names[merged_anndata.obs['condition'] == str(sample_name)].tolist()
    print(f"Merged cells (full) for sample {sample_name}: {merged_cells_full[:5]}... ({len(merged_cells_full)} total)")
    
    # Clean AnnData cell names
    merged_cells_cleaned = [cell.split('-')[0] for cell in merged_cells_full]
    print(f"Merged cells (cleaned) for sample {sample_name}: {merged_cells_cleaned[:5]}... ({len(merged_cells_cleaned)} total)")
    
    # Find common cleaned cell barcodes
    common_cleaned = np.intersect1d(loom_cells_cleaned, merged_cells_cleaned)
    print(f"Common cells for sample {sample_name}: {len(common_cleaned)}")
    
    if len(common_cleaned) == 0:
        print(f"No matching cells found for sample {sample_name}.")
        loom.close()
        return None, None, None
    
    # Build list of full AnnData cell names that match the common cleaned barcodes
    cells_to_keep = [cell_full for cell_full in merged_cells_full if cell_full.split('-')[0] in common_cleaned]
    
    # Subset Loom data to common cells
    loom_cells_idx = np.isin(loom_cells_cleaned, common_cleaned)
    loom_cells_idx_int = np.where(loom_cells_idx)[0]
    spliced = loom.layers['spliced'][:, loom_cells_idx_int]
    unspliced = loom.layers['unspliced'][:, loom_cells_idx_int]
    
    print(f"Subsetted Loom data from {len(loom_cells_cleaned)} → {len(common_cleaned)} cells for sample {sample_name}")
    
    # Handle genes
    loom_genes = loom.ra['Gene']
    merged_genes = merged_anndata.var_names.tolist()
    loom_genes_cleaned = [gene.split('_')[0] for gene in loom_genes]
    common_genes = np.intersect1d(loom_genes_cleaned, merged_genes)
    print(f"Common genes for sample {sample_name}: {len(common_genes)}")
    
    if len(common_genes) == 0:
        print(f"No common genes found for sample {sample_name}.")
        loom.close()
        return None, None, None
    
    # Subset Loom matrices to include only common genes
    loom_genes_idx = np.isin(loom_genes_cleaned, common_genes)
    loom_genes_idx_int = np.where(loom_genes_idx)[0]
    spliced = spliced[loom_genes_idx_int, :]
    unspliced = unspliced[loom_genes_idx_int, :]
    
    # Transpose Loom matrices to shape (cells, genes)
    spliced = spliced.T
    unspliced = unspliced.T
    
    loom.close()
    
    # Return the velocity matrices and cell names, NOT the modified AnnData
    return spliced, unspliced, cells_to_keep

In [12]:
# Process each sample separately
combined_spliced, combined_unspliced = [], []
processed_anndatas = []
cell_subsets = {}

for sample_name in samples:
    # Construct loom file path directly from sample name
    loom_file = f"/data/ebaird/scRNAseq/SCENTINELsep24/cellranger/{sample_name}/velocyto/{sample_name}.loom"
    
    spliced, unspliced, cells_to_keep = add_velocity_to_anndata(merged_adata, loom_file, sample_name)
    if spliced is not None:
        combined_spliced.append(spliced)
        combined_unspliced.append(unspliced)
        cell_subsets[sample_name] = cells_to_keep
        print(f"Finished processing sample {sample_name} with {spliced.shape[0]} cells.")
    else:
        print(f"Skipping sample {sample_name} due to errors.")

In [13]:
# Get all cells that have velocity data
all_velocity_cells = []
for cells in cell_subsets.values():
    all_velocity_cells.extend(cells)

# Subset the original AnnData to only include cells with velocity data
final_anndata = merged_adata[all_velocity_cells, :]

# Concatenate the velocity matrices
combined_spliced_mat = np.concatenate(combined_spliced, axis=0)
combined_unspliced_mat = np.concatenate(combined_unspliced, axis=0)

print(f"Final combined spliced shape: {combined_spliced_mat.shape}")
print(f"Final combined unspliced shape: {combined_unspliced_mat.shape}")
print(f"Final merged AnnData shape: {final_anndata.shape}")

# Now they should match
if combined_spliced_mat.shape[0] != final_anndata.n_obs:
    raise ValueError(f"Mismatch in cell count: spliced matrix has {combined_spliced_mat.shape[0]} cells but final AnnData has {final_anndata.n_obs}.")

# Assign the velocity layers
final_anndata.layers['spliced'] = combined_spliced_mat
final_anndata.layers['unspliced'] = combined_unspliced_mat

# Save the updated merged AnnData.
final_anndata.write(os.path.join(repDir, "merged_anndata_with_velocity.h5ad"))


In [14]:
adata = final_anndata

In [15]:
adata.obs['seurat_clusters'] = adata.obs['seurat_clusters'].astype('category')

scv.pl.proportions(adata, groupby='seurat_clusters',
                   save=os.path.join(figDir, 'proportions_gal.pdf'))

In [15]:
# adata_Gal = adata[adata.obs['genotype'] == str("gal")].copy()
# adata_Flp = adata[adata.obs['genotype'] == str("flp")].copy()

In [16]:
# adata = adata_Gal

In [ ]:
# print("Computing neighborhood graph using Seurat PCA...")
# sc.pp.neighbors(adata, use_rep='X_pca', n_neighbors=30, n_pcs=min(30, seurat_pca.shape[1]))

# sc.tl.paga(adata, groups='seurat_clusters')
# sc.pl.paga(adata, threshold=0.1)
# sc.tl.umap(adata, init_pos='paga')
# sc.pl.umap(adata, color=['seurat_clusters'], legend_loc='on data')

In [16]:
scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=2000)
scv.pp.moments(adata, n_pcs=30, n_neighbors=30)
scv.tl.recover_dynamics(adata)
scv.tl.velocity(adata, mode='dynamical', diff_kinetics=True)
scv.tl.velocity_graph(adata)

In [17]:
scv.pl.velocity_embedding_stream(adata, basis="umap", color = 'seurat_clusters', save=os.path.join(figDir, 'embedding_stream.pdf'))

In [18]:
# adata.write(repDir + 'dynamical_diffkinetics_results.h5ad')
# adata = sc.read(repDir + 'dynamical_diffkinetics_results.h5ad')

In [32]:
df = adata.var
df = df[(df['fit_likelihood'] > .1) & df['velocity_genes'] == True]

kwargs = dict(xscale='log', fontsize=16)
with scv.GridSpec(ncols=3) as pl:
    pl.hist(df['fit_alpha'], xlabel='transcription rate', **kwargs)
    pl.hist(df['fit_beta'] * df['fit_scaling'], xlabel='splicing rate', xticks=[.1, .4, 1], **kwargs)
    pl.hist(df['fit_gamma'], xlabel='degradation rate', xticks=[.1, .4, 1], **kwargs)

scv.get_df(adata, 'fit*', dropna=True).head()

In [38]:
scv.tl.latent_time(adata)
scv.pl.scatter(adata, color='latent_time', color_map='gnuplot', size=80, save=os.path.join(figDir, 'latent_time.pdf'))

top_genes = adata.var['fit_likelihood'].sort_values(ascending=False).index[:300]
scv.pl.heatmap(adata, var_names=top_genes, sortby='latent_time', col_color='seurat_clusters', n_convolve=200, save=os.path.join(figDir, 'latent_time_htmp.pdf'))

In [34]:
top_genes = adata.var['fit_likelihood'].sort_values(ascending=False).index
scv.pl.scatter(adata, basis=top_genes[:15], ncols=5, frameon=False, color = 'seurat_clusters', save=os.path.join(figDir, 'top_genes.pdf'))

In [35]:
scv.pl.scatter(adata, x='latent_time', y=top_genes[:15], ncols=5, frameon=False, color='seurat_clusters', save=os.path.join(figDir, 'top_genes_latent_time.pdf'))

In [37]:
scv.tl.rank_dynamical_genes(adata, groupby='seurat_clusters')
df = scv.get_df(adata, 'rank_dynamical_genes/names')
df.head(15)

In [41]:
scv.pl.velocity(adata, top_genes[:10], ncols=2, save=os.path.join(figDir, 'top_genes_velocity_plus.pdf'))

In [40]:
scv.tl.velocity_confidence(adata)
keys = 'velocity_length', 'velocity_confidence'
scv.pl.scatter(adata, c=keys, cmap='coolwarm', perc=[5, 95], save=os.path.join(figDir, 'velocity_length_confidence.pdf'))

In [42]:
scv.pl.velocity_graph(adata, threshold=.1, color='seurat_clusters', save=os.path.join(figDir, 'velocity_graph.pdf'))

In [44]:
x, y = scv.utils.get_cell_transitions(adata, basis='umap', starting_cell=600)
ax = scv.pl.velocity_graph(adata, c='lightgrey', edge_width=.05, show=False)
ax = scv.pl.scatter(adata, x=x, y=y, s=120, c='ascending', cmap='gnuplot', ax=ax)

In [18]:
import cellrank as cr

In [19]:
vk = cr.kernels.VelocityKernel(adata)

In [20]:
vk.compute_transition_matrix()

In [21]:
ck = cr.kernels.ConnectivityKernel(adata)
ck.compute_transition_matrix()

combined_kernel = 0.8 * vk + 0.2 * ck

In [22]:
print(combined_kernel)

In [23]:
vk.plot_projection()

In [ ]:
vk.plot_random_walks(start_ixs={"seurat_clusters": 5
                                }, max_iter=200, seed=0)

In [26]:
g = cr.estimators.GPCCA(vk)
print(g)

In [27]:
g.fit(cluster_key="clusters", n_states=[4, 12])
g.plot_macrostates(which="all", discrete=True, legend_loc="right", s=100)

In [28]:
from cellrank.kernels import CytoTRACEKernel

ctk = CytoTRACEKernel(adata).compute_cytotrace()

In [31]:
sc.pl.embedding(
    adata,
    color=["ct_pseudotime"],
    basis="umap",
    color_map="gnuplot2",
)

# Save plot
sc.pl.embedding(
    adata,
    color=["ct_pseudotime"],
    basis="umap",
    color_map="gnuplot2",
    save=os.path.join('cytotrace_pseudotime.pdf')
)

In [71]:
sc.pl.violin(adata, keys=["ct_pseudotime"], groupby="seurat_clusters", rotation=90)

In [72]:
ctk.compute_transition_matrix(threshold_scheme="soft", nu=0.5)

In [74]:
ctk.plot_projection(basis="umap", color="seurat_clusters", legend_loc="right")

In [78]:
ctk.plot_random_walks(
    n_sims=100,
    start_ixs={"seurat_clusters": 0},
    basis="umap",
    color="seurat_clusters",
    legend_loc="right",
    seed=1,
)